# 01. シミュレータで障害物回避を学習する (toio × ATOM Mate for toio)

toio + ATOM Mate for toio のロボットが、**マット上のゴールへ障害物を避けながら移動する** 方策を、
2D シミュレータ上の強化学習 (PPO) で学習します。学習した方策は `02_run_on_real_robot.ipynb` で実機に載せます。

- **観測**: ToF 距離、ゴールの距離と方位、マット上の位置と向き、前回の行動 (直近 0.8 秒分)、障害物メモリ (周囲 8 方向)
- **行動**: 前進・旋回 → toio の左右モーター速度指示値
- **sim2real**: 実機の物理定数・通信遅延・センサーノイズなどを **エピソードごとにランダム化** (ドメインランダマイゼーション)

> ランタイムは **CPU で十分** です (小さな MLP なので GPU にしても速くなりません)。

## 1. セットアップ

In [ ]:
# --- プロジェクトの場所 ---
# このフォルダ (robot-test) を Google ドライブの「マイドライブ」直下にアップロードしておく
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/robot-test'  # ← アップロード先に合わせて変更

# GitHub に置いた場合はドライブの代わりに clone してもよい (学習結果の保存先はドライブを推奨)
# !git clone https://github.com/<user>/<repo>.git /content/robot-test
# PROJECT_DIR = '/content/robot-test'

import os, sys
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
%pip install -q -r requirements.txt

## 2. 設定

主な項目 (すべて `toio_sim2real/config.py` の `EnvConfig`):

| 項目 | 既定値 | 意味 |
|---|---|---|
| `mat` | `"toio_collection"` | 使うマット。簡易プレイマットなら `"simple"` |
| `max_forward_unit` | 45 | 前進の最大速度指示値 (約 126 mm/s) |
| `tof_max_range_mm` | 400 | これ以上の ToF 値は「障害物なし」 |
| `dr.latency_s` | (0.08, 0.45) | 学習時にランダム化する通信遅延の範囲 [秒] |
| `obstacle_count` | (1, 5) | 1 エピソードの障害物の数 |

実機のキャリブレーション (02 ノートブック) で DR 範囲外の値が出た場合は、保存された `runs/suggested_config.json` を読み込んで学習し直します。

In [ ]:
import json
from toio_sim2real import EnvConfig

cfg = EnvConfig(mat="toio_collection")  # 簡易プレイマットなら "simple"

# 例: 実機キャリブレーションの結果を反映した設定を使う
# cfg = EnvConfig.load(f"{PROJECT_DIR}/runs/suggested_config.json")

print(f"マット: {cfg.mat}  {cfg.mat_spec.width_mm:.0f} x {cfg.mat_spec.height_mm:.0f} mm")
print("DR:", json.dumps(cfg.to_dict()["dr"], ensure_ascii=False))

## 3. シミュレータを見てみる (ルールベース制御)

まずは学習なしの単純な制御 (ToF が近ければ右旋回、そうでなければゴール方向へ) で、シミュレータの様子を確認します。
オレンジの扇形が ToF センサーの視野と測定距離、灰色が障害物、赤い星がゴールです。

In [ ]:
from IPython.display import HTML
from toio_sim2real.env import ToioAvoidEnv
from toio_sim2real.evaluate import evaluate, run_episode, latency_sweep
from toio_sim2real.policies import RuleBasedPolicy
from toio_sim2real import viz

rule = RuleBasedPolicy(cfg)
res = run_episode(ToioAvoidEnv(cfg), rule, seed=1, record=True)
HTML(viz.animate_episode(res, cfg, max_frames=300).to_jshtml())

In [ ]:
print("ルールベース:", evaluate(rule, cfg, n_episodes=30))

## 4. 学習

PPO で学習します。既定の 200 万ステップは、Colab の CPU ランタイムで 1 時間前後が目安です。
学習結果は `RUN_DIR` に保存されます (`model.zip` と `env_config.json`)。25 万ステップごとのチェックポイントも
`RUN_DIR/checkpoints/` に残ります。セッションが切れた場合は次のセルの `resume_from` にチェックポイントを指定して再開できます。
途中経過は TensorBoard で確認できます。`toio/goals_per_episode` (ゴール到達数) と
`toio/collision_rate` (衝突で終わったエピソードの割合) が主な指標です。

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

In [ ]:
from toio_sim2real.train import train

RUN_DIR = f"{PROJECT_DIR}/runs/ppo_avoid"
TOTAL_STEPS = 2_000_000

model = train(cfg, RUN_DIR, total_timesteps=TOTAL_STEPS, n_envs=8)

### (任意) 続きから学習 / 実機キャリブレーション後の追加学習

`resume_from` に既存のモデルやチェックポイントを指定すると、そこから学習を続けます。
02 ノートブックで DR 範囲外の値が出た場合は、保存された設定で追加学習します。

In [ ]:
# 例 1: 切断されたので 100 万ステップのチェックポイントから残りを学習
# model = train(cfg, RUN_DIR, total_timesteps=1_000_000,
#               resume_from=f"{RUN_DIR}/checkpoints/ppo_1000000_steps.zip")

# 例 2: 実機キャリブレーション結果の設定で追加学習 (結果は別フォルダに保存)
# cfg = EnvConfig.load(f"{PROJECT_DIR}/runs/suggested_config.json")
# model = train(cfg, f"{PROJECT_DIR}/runs/ppo_avoid_ft", total_timesteps=500_000,
#               resume_from=f"{PROJECT_DIR}/runs/ppo_avoid/model.zip")

## 5. 評価

同じ乱数シードの 30 エピソード (各 60 秒) で、学習した方策とルールベースを比べます。

- **DR あり**: 実機を想定したランダムな物理・遅延・ノイズ
- **DR なし**: 公称値・ノイズなし

In [ ]:
import pandas as pd
from toio_sim2real.policies import load_trained

model, cfg = load_trained(RUN_DIR)
rows = {}
for name, pol in [("PPO", model), ("ルールベース", RuleBasedPolicy(cfg))]:
    rows[f"{name} (DR あり)"] = evaluate(pol, cfg, n_episodes=30)
    rows[f"{name} (DR なし)"] = evaluate(pol, cfg, n_episodes=30, dr=False)
pd.DataFrame(rows).T.round(2)

In [ ]:
res = run_episode(ToioAvoidEnv(cfg), model, seed=1, record=True)
HTML(viz.animate_episode(res, cfg, max_frames=400).to_jshtml())

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
env = ToioAvoidEnv(cfg)
for i, ax in enumerate(axes.flat):
    viz.plot_episode(run_episode(env, model, seed=100 + i, record=True), cfg, ax=ax)
plt.tight_layout()

## 6. 通信遅延に対する頑健性

Colab ⇔ MQTT ブローカー ⇔ ATOM の往復遅延は環境によって大きく変わります。
遅延を固定値にして評価し、学習時の DR 範囲 (灰色) の外でどれだけ性能が落ちるかを確認します。
02 ノートブックで測った実機の遅延がこの範囲に入っていれば安心です。

In [ ]:
lats = (0.05, 0.15, 0.3, 0.45, 0.6, 0.8)
sweep_ppo = latency_sweep(model, cfg, lats, n_episodes=15)
sweep_rule = latency_sweep(RuleBasedPolicy(cfg), cfg, lats, n_episodes=15)
viz.plot_latency_sweep([sweep_ppo, sweep_rule], ["PPO", "rule-based"], dr_range=cfg.dr.latency_s);

## 次へ

`notebooks/02_run_on_real_robot.ipynb` で、この `RUN_DIR` のモデルを実機で動かします。